# Importations

In [18]:
import re
import csv
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WRAP = 2 ** 32  # Période de débordement du compteur run-time (microsecondes)

# Affectation des tâches aux cœurs
CORE1 = {"audio", "ui", "IDLE1", "ipc1"}
CORE0 = {"maint", "esp_timer", "ipc0", "IDLE0",
         "BTC_TASK", "BTU_TASK", "btController", "hciT"}

# Préparation des données

In [19]:
def strip_ansi(text):
    """Retire les séquences d'échappement ANSI et les retours chariot."""
    text = re.sub(r"\x1b\[[0-9;]*m", "", text)
    return text.replace("\r", "")

def parse_blocks(path):
    """Extrait la liste des blocs diag { t, tasks, rt, heap } du log."""
    # Note: Dans un notebook, assurez-vous que le fichier est dans le même dossier
    # ou fournissez le chemin absolu.
    try:
        content = open(path, encoding="utf-8", errors="replace").read()
    except FileNotFoundError:
        print(f"Erreur: Le fichier '{path}' est introuvable.")
        return []

    lines = strip_ansi(content).split("\n")
    blocks, cur = [], None

    for ln in lines:
        if re.match(r"^I \(\d+\) diag: ={4,} diagnostics ={4,}", ln):
            m = re.match(r"^I \((\d+)\)", ln)
            if m:
                cur = {"t": int(m.group(1)), "tasks": {}, "rt": {}, "heap": {}}
                blocks.append(cur)
            continue

        if cur is None:
            continue

        # Ligne de tâche
        m = re.match(
            r"^I \(\d+\) diag: (\S+)\s+(\S+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)%", ln)
        if m and m.group(1) not in ("task", "all", "run", "internal", "psram"):
            cur["tasks"][m.group(1)] = dict(
                core=m.group(2), prio=int(m.group(3)), stack=int(m.group(4)),
                hwm=int(m.group(5)), used=int(m.group(6)), pct=int(m.group(7)))
            continue

        # Ligne de run-time
        m = re.match(r"^(\S+)\s+\t(\d+)\t\t<?\d+%$", ln)
        if m:
            cur["rt"][m.group(1)] = int(m.group(2))
            continue

        # Heap interne
        m = re.match(r"^I \(\d+\) diag: internal\s+free\s+(\d+) B\s+min free ever\s+(\d+) B", ln)
        if m:
            cur["heap"]["int_free"] = int(m.group(1))
            cur["heap"]["int_min"] = int(m.group(2))
            continue

        # PSRAM
        m = re.match(r"^I \(\d+\) diag: psram\s+free\s+(\d+) B\s+min free ever\s+(\d+) B", ln)
        if m:
            cur["heap"]["ps_free"] = int(m.group(1))
            cur["heap"]["ps_min"] = int(m.group(2))

    return [b for b in blocks if b["tasks"]]

# Calcul charge CPU

In [20]:
def compute_load(blocks):
    """Charge par tâche (delta corrigé du débordement) pour chaque intervalle."""
    if not blocks:
        return [], []

    names = sorted({n for b in blocks for n in b["rt"]})
    rows = []

    for a, b in zip(blocks, blocks[1:]):
        dt = (b["t"] - a["t"]) * 1000.0  # ms -> us
        if dt <= 0:
            continue

        r = {"t_s": round(b["t"] / 1000.0, 1)}
        for n in names:
            if n in a["rt"] and n in b["rt"]:
                d = (b["rt"][n] - a["rt"][n]) % WRAP
            elif n in b["rt"]:
                d = b["rt"][n]
            else:
                d = 0
            r[n] = round(100.0 * d / dt, 2)

        r["charge_coeur1"] = round(sum(r.get(n, 0) for n in CORE1 if n != "IDLE1"), 2)
        r["charge_coeur0"] = round(sum(r.get(n, 0) for n in CORE0 if n != "IDLE0"), 2)
        rows.append(r)

    return names, rows

# Création CSV

In [21]:
# --- CONFIGURATION ---
LOG_FILE = "../assets/data/campagne_0713_1356.log"  # <--- MODIFIEZ CECI

# Phases de la campagne (en minutes)
PHASES = [
    (11 / 60,    150 / 60,   "#eeeeee"),  # repos
    (150 / 60,   2700 / 60,  "#dce9f5"),  # filaire WAV/MP3
    (2700 / 60,  4650 / 60,  "#f5e2dc"),  # streaming Bluetooth A2DP
    (4650 / 60,  12879 / 60, "#dce9f5"),  # retour filaire
]

# Exécution du parsing et du calcul
blocks = parse_blocks(LOG_FILE)

if blocks:
    names, rows = compute_load(blocks)

    # --- BLOC DE SORTIE CSV ---
    cols = ["t_s"] + names + ["charge_coeur0", "charge_coeur1"]
    csv_path = "../assets/data/charge_cpu.csv"

    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)

    print(f"✅ Fichier créé : {csv_path}")
    print(f"   Données traitées : {len(rows)} intervalles")
else:
    print("❌ Aucun bloc diag trouvé. Vérifiez le nom du fichier log.")

✅ Fichier créé : ../assets/data/charge_cpu.csv
   Données traitées : 1258 intervalles


# Graphique charge CPU

In [22]:
if blocks and rows:
    t = [r["t_s"] / 60 for r in rows]
    c0 = [r["charge_coeur0"] for r in rows]
    c1 = [r["charge_coeur1"] for r in rows]

    fig, ax = plt.subplots(figsize=(9, 3.6))

    # Fond des phases
    for a, b, col in PHASES:
        ax.axvspan(a, b, color=col, zorder=0)

    ax.plot(t, c1, lw=0.7, color="#1f4e79", label="cœur 1 (audio, interface)")
    ax.plot(t, c0, lw=0.7, color="#a6371f", label="cœur 0 (Bluetooth, maintenance)")

    ax.set_xlim(0, t[-1])
    ax.set_ylim(0, 100)
    ax.set_xlabel("Temps (minutes)")
    ax.set_ylabel("Charge processeur (%)")
    ax.legend(loc="upper left", fontsize=8, framealpha=0.9)

    ax.annotate("pile Bluetooth\nsur cœur 0", xy=(60, 68), xytext=(60, 92),
                fontsize=7, ha="center", color="#a6371f",
                arrowprops=dict(arrowstyle="->", color="#a6371f", lw=0.7))

    ax.grid(True, axis="y", lw=0.3, alpha=0.4)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

    fig.tight_layout()

    # --- BLOC DE SORTIE IMAGES ---
    pdf_path = "../assets/figures/charge_cpu.pdf"

    fig.savefig(pdf_path)

    print(f"✅ Fichiers créés : {pdf_path}")

    # Affichage dans le notebook
    plt.show()
else:
    print("Aucune donnée à afficher.")

✅ Fichiers créés : ../assets/figures/charge_cpu.pdf


/tmp/ipykernel_3255/2790008533.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Graphique stackup

In [23]:
if blocks:
    pts = [(b["t"] / 1000 / 60, b["heap"]["int_free"] / 1024)
           for b in blocks if "int_free" in b["heap"]]

    if pts:
        t = [p[0] for p in pts]
        h = [p[1] for p in pts]

        fig, ax = plt.subplots(figsize=(9, 2.6))

        for a, b, col in PHASES:
            ax.axvspan(a, b, color=col, zorder=0)

        ax.plot(t, h, lw=0.7, color="#1f4e79")

        ax.set_xlim(0, t[-1])
        ax.set_ylim(0, max(h) * 1.1)
        ax.set_xlabel("Temps (minutes)")
        ax.set_ylabel("Tas interne libre (kio)")
        ax.grid(True, axis="y", lw=0.3, alpha=0.4)

        for s in ("top", "right"):
            ax.spines[s].set_visible(False)

        fig.tight_layout()

        # --- BLOC DE SORTIE IMAGES ---
        pdf_path = "../assets/figures/tas_interne.pdf"

        fig.savefig(pdf_path)

        print(f"✅ Fichiers créés : {pdf_path}")
        print(f"   Tas interne libre : min {min(h):.1f} kio, max {max(h):.1f} kio")

        # Affichage dans le notebook
        plt.show()
    else:
        print("Aucune donnée de tas interne trouvée.")
else:
    print("Aucun bloc disponible.")

✅ Fichiers créés : ../assets/figures/tas_interne.pdf
   Tas interne libre : min 69.0 kio, max 140.7 kio


/tmp/ipykernel_3255/2405279702.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Résumé

In [24]:
if blocks and rows:
    print("\n=== RÉCAPITULATIF ===")
    dur_h = (blocks[-1]["t"] - blocks[0]["t"]) / 3.6e6
    print(f"Durée : {dur_h:.2f} h, {len(blocks)} relevés")

    last = blocks[-1]["tasks"]
    print("\nPiles (HWM minimal sur la campagne) :")

    tasks_check = ["audio", "ui", "input", "maint"]
    for n in tasks_check:
        if n in last:
            hwms = [b["tasks"][n]["hwm"] for b in blocks if n in b["tasks"]]
            stk = last[n]["stack"]
            min_hwm = min(hwms) if hwms else 0
            occup_max = 100 * (stk - min_hwm) / stk if stk > 0 else 0
            print(f"  {n:6s} pile {stk:5d} o  marge min {min_hwm:5d} o  "
                  f"occup max {occup_max:.0f}%")
        else:
            print(f"  {n:6s} (non trouvée)")

    i0 = [r["charge_coeur0"] for r in rows]
    i1 = [r["charge_coeur1"] for r in rows]
    print(f"\nCharge crête cœur 0 : {max(i0):.1f}%   cœur 1 : {max(i1):.1f}%")
else:
    print("Impossible de générer le résumé (données manquantes).")


=== RÉCAPITULATIF ===
Durée : 3.57 h, 1259 relevés

Piles (HWM minimal sur la campagne) :
  audio  pile  4096 o  marge min  1544 o  occup max 62%
  ui     pile  8192 o  marge min  4348 o  occup max 47%
  input  pile  3072 o  marge min  2076 o  occup max 32%
  maint  pile  3072 o  marge min  1708 o  occup max 44%

Charge crête cœur 0 : 81.2%   cœur 1 : 63.7%
